# Worst cases of a trained SiameseNet

Inputs, made on the cluster by `scripts/worst_cases.py` (run automatically at the end of `scripts/train_siamese.sbatch`):
- `RUN/worst/errors.csv`: every val lesion, worst first: error, true offset `t*`, predicted offset `p*` (mm, x,y,z),
  FU lesion radius, `peak_prob` (highest cell probability of the heatmap: low = flat or several peaks).
- `RUN/worst/heatmaps.npz`: the 32³ heatmap (4 mm cells) of the top N lesions.

CT and masks are read from the lesion npz files in `DATA` (the same fixed-jitter val patches the model was scored on).
Offsets → voxels: `C + offset / 2 mm`, reordered from (x,y,z) to array order (z,y,x), as in `example_lesions.ipynb`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ipywidgets import Dropdown, IntSlider, ToggleButtons, interact

RUN = Path("../runs/3279130")  # job id of the run to inspect
DATA = Path("/gpfs/helios/projects/ml_project_lesion_tracking/data/Longitudinal-CT-processed-team2/lesions")
SPACING, C = 2.0, 31.5  # mm per voxel; patch centre voxel index

df = pd.read_csv(RUN / "worst" / "errors.csv")
df["target_mm"] = np.linalg.norm(df[["tx", "ty", "tz"]], axis=1)
maps = np.load(RUN / "worst" / "heatmaps.npz")
top = df[df.lesion.isin(maps.files)]
print(f"{len(df)} lesions, mean {df.err_mm.mean():.2f} mm, median {df.err_mm.median():.2f} mm, "
      f"{(df.err_mm > 10).mean():.1%} over 10 mm")
top

## 1. Where do the big errors come from?

Is the error tied to lesion size, to how sure the model was (`peak_prob`), or to how far the window was off (`|target|`)?

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 3.8))
ax[0].hist(df.err_mm, bins=40)
ax[0].axvline(df.err_mm.median(), color="k", ls="--", label="median")
ax[0].set(xlabel="error [mm]", ylabel="lesions", title="error distribution")
ax[0].legend()
for a, col in zip(ax[1:], ["fu_radius_mm", "peak_prob", "target_mm"]):
    a.scatter(df[col], df.err_mm, s=8, alpha=0.6)
    a.scatter(top[col], top.err_mm, s=14, color="r", label=f"top {len(top)}")
    a.set(xlabel=col, ylabel="error [mm]", title=f"error vs {col}")
ax[1].set_xscale("log")
ax[1].legend()
fig.tight_layout()

## 2. Slice viewer

Pick a lesion (worst first), a view and a slice; the slice starts at the true FU lesion.
Green `+` = true FU lesion, red `×` = prediction, cyan `+` = window centre. Yellow contour = lesion mask.
Markers are drawn on every slice; the title says which slice each point is on.
The BL panel cuts the same slice; the BL lesion itself is at the patch centre (slice 31–32).

In [ ]:
VIEWS = {"axial": 0, "coronal": 1, "sagittal": 2}  # array axis that is cut: z, y, x


def to_vox(xyz_mm):
    """(x,y,z) mm offset from the patch centre -> (z,y,x) voxel position."""
    return (C + np.asarray(xyz_mm, dtype=float) / SPACING)[::-1]


def cut(a, axis, i):
    return np.take(a, i, axis=axis).astype(np.float32)


def show(lesion, view, slice_):
    r = df.set_index("lesion").loc[lesion]
    d = np.load(DATA / f"{lesion}.npz")
    hm = maps[lesion].astype(np.float32).repeat(2, 0).repeat(2, 1).repeat(2, 2)  # cell i -> voxels 2i, 2i+1
    k = VIEWS[view]
    rows, cols = [j for j in range(3) if j != k]  # in-plane array axes
    true, pred = to_vox(r[["tx", "ty", "tz"]]), to_vox(r[["px", "py", "pz"]])
    fig, axs = plt.subplots(1, 3, figsize=(16, 5.5))
    panels = [("BL + BL mask", d["bl"], d["bl_mask"], None),
              ("FU + FU mask", d["fu"], d["fu_mask"], None),
              ("FU + heatmap", d["fu"], None, hm)]
    for a, (title, img, mask, heat) in zip(axs, panels):
        a.imshow(cut(img, k, slice_), cmap="gray", vmin=0, vmax=1, origin="lower")
        if mask is not None and cut(mask, k, slice_).any():
            a.contour(cut(mask, k, slice_), levels=[0.5], colors="yellow", linewidths=1)
        if heat is not None:
            h = cut(heat, k, slice_)
            a.imshow(np.ma.masked_less(h, 0.01 * heat.max()), cmap="hot", alpha=0.5, origin="lower",
                     vmin=0, vmax=heat.max())
        if title.startswith("FU"):
            a.plot(true[cols], true[rows], "g+", ms=16, mew=2)
            a.plot(pred[cols], pred[rows], "rx", ms=12, mew=2)
            a.plot(C, C, "c+", ms=10, mew=1)
        else:
            a.plot(C, C, "g+", ms=16, mew=2)  # BL lesion is at the patch centre
        a.set_title(title)
    fig.suptitle(f"{lesion}: error {r.err_mm:.1f} mm, |target| {r.target_mm:.1f} mm, radius {r.fu_radius_mm:.1f} mm, "
                 f"peak_prob {r.peak_prob:.3f} | {view} slice {slice_}, true on {true[k]:.0f}, pred on {pred[k]:.0f}")
    fig.tight_layout()
    plt.show()


lesion_w = Dropdown(options=[(f"{i + 1}. {n} ({e:.1f} mm)", n) for i, (n, e) in enumerate(zip(top.lesion, top.err_mm))],
                    description="lesion")
view_w = ToggleButtons(options=list(VIEWS), description="view")
slice_w = IntSlider(min=0, max=63, description="slice", continuous_update=False)


def jump(*_):  # move the slider to the slice through the true lesion
    r = df.set_index("lesion").loc[lesion_w.value]
    slice_w.value = int(np.clip(round(to_vox(r[["tx", "ty", "tz"]])[VIEWS[view_w.value]]), 0, 63))


lesion_w.observe(jump, "value")
view_w.observe(jump, "value")
jump()
interact(show, lesion=lesion_w, view=view_w, slice_=slice_w);